# E-Commerce Customer Churn Prediction & Risk Analytics using RFM and Machine Learning
### Rohit Dey · AICTE | IBM SkillsBuild Data Analytics with AI Academic Internship 2026
**BharatCares in association with AICTE and IBM SkillsBuild**

## 01 Project Overview
**Question:** Which existing customers may make no purchase during the next 180 days, and how could risk scores inform retention decisions?

Raw Data → Clean Data → EDA → Business Insights → Prediction → Dashboard → Business Decision.

**Source exception approved by the student:** only the official practice dataset's 95-page PDF export was available. No Kaggle or supermarket data are used. Original Excel sheet names, formulas and native cell types cannot be verified. A coordinate-validated reconstruction is saved locally as Excel, explicitly labelled reconstructed—not the original workbook.

**Reproducibility:** run all cells in order. Put the official PDF in `data/` beside this notebook (or upload it into the Colab working directory). No network data download is performed. Outputs contain coded customer identifiers, not names, email addresses or chat participants. Keep customer-level outputs private unless publication is permitted.

**Academic integrity:** metrics and findings below are computed, not copied from teaching examples. AI assisted code/document preparation; the student should review and understand the work before submission.

## 02 Import Libraries
PDF reconstruction is isolated from the beginner-friendly pandas and scikit-learn analysis. A fixed random seed makes the experiment repeatable.

In [ ]:
# Import only libraries used by the analysis and dashboard.
from pathlib import Path
import re, json, hashlib, platform
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.offline import get_plotlyjs
from pypdf import PdfReader
from IPython.display import display, Markdown, HTML
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay, brier_score_loss)
SEED = 42
ROOT = Path.cwd()
OUT = ROOT / 'outputs'
CHARTS = OUT / 'charts'
DASH = ROOT / 'dashboard'
for directory in [ROOT/'data', OUT, CHARTS, DASH]: directory.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13})
pd.set_option('display.max_columns', 14)
print('Python:', platform.python_version())
print({p: version(p) for p in ['pandas','numpy','scikit-learn','pypdf','plotly']})

## 03 Load Dataset
### PDF reconstruction and verification
The inspected export places seven transaction fields on pages 1–44 and Revenue/Profit on pages 45–88. Pages 89–95 are classroom prompts, not records.

The helper decodes the PDF's original text objects (including full product names), assigns each cell to its printed column, and matches identical vertical row coordinates across paired pages. It checks headers, every paired baseline, ordinary text extraction and duplicate fingerprints. It **fails instead of guessing** if the layout differs. Empty cells remain missing. Coordinates establish export consistency, not independent verification against the unavailable Excel original.

The generated workbook has a **new** sheet named `Reconstructed_Transactions`. Its name is not claimed to be an original sheet name. Excel loading and sheet identification below operate on this derived workbook only.

In [ ]:
# Advanced source adapter: no cleaning or invented values occurs here.
def extract_official_pdf(path):
    """Decode original PDF text cells by position; never infer absent cell values.
    Specific to the inspected 95-page official export. Fail closed if layout differs.
    """
    import re
    import hashlib
    import pandas as pd
    from pypdf import PdfReader
    reader = PdfReader(path)
    assert len(reader.pages) == 95, 'Unexpected export; inspect layout before continuing.'
    columns = ['Order_ID', 'Order_Date', 'Customer_ID', 'Product', 'Category', 'Region', 'Quantity', 'Revenue', 'Profit']
    def cells(page):
        maps = {}
        for name, ref in page['/Resources']['/Font'].items():
            font = ref.get_object()
            cmap = font['/ToUnicode'].get_data().decode('ascii')
            mapping = {}
            # This export uses two-byte glyphs and simple sequential Unicode ranges.
            for block in re.findall(r'beginbfrange(.*?)endbfrange', cmap, re.S):
                assert '[' not in block, 'Unsupported font mapping.'
                for a, b, c in re.findall(r'<([0-9A-Fa-f]+)>\s*<([0-9A-Fa-f]+)>\s*<([0-9A-Fa-f]+)>', block):
                    for n in range(int(a,16), int(b,16)+1):
                        mapping[n] = chr(int(c,16) + n - int(a,16))
            assert mapping
            maps[name] = mapping
        result = {}; current = [None]
        def visitor(op, args, cm, tm):
            if op == b'Tf': current[0] = args[0]
            if op not in (b'Tj', b'TJ'): return
            assert cm == [1,0,0,1,0,0], 'Unexpected coordinate transform.'
            parts = args[0] if op == b'TJ' else [args[0]]
            text = ''
            for part in parts:
                if isinstance(part, bytes):
                    assert len(part) % 2 == 0
                    text += ''.join(maps[current[0]][int.from_bytes(part[i:i+2], 'big')] for i in range(0,len(part),2))
                elif isinstance(part, str): text += part
            if text.strip():
                y, x = round(tm[5],2), round(tm[4],2)
                result.setdefault(y, []).append((x,text.strip()))
        page.extract_text(visitor_operand_before=visitor)
        return result
    records=[]; provenance=[]; audit=[]
    for i in range(44):
        left, right = cells(reader.pages[i]), cells(reader.pages[i+44])
        if i == 0:
            assert [v for x,v in sorted(left[max(left)])] == columns[:7]
            assert [v for x,v in sorted(right[max(right)])] == columns[7:]
            del left[max(left)]; del right[max(right)]
        # Identical physical row baselines on paired pages establish positional alignment.
        assert set(left) == set(right), f'Unmatched row coordinates: pages {i+1}/{i+45}'
        for y in sorted(left, reverse=True):
            row=['']*9
            for x, value in left[y]:
                col=next((j for j,b in enumerate([122,191,260,330,399,468,540]) if x < b),None)
                assert col is not None and not row[col], 'Cell collision.'
                row[col]=value
            for x,value in right[y]:
                col=7 if x<122 else 8
                assert x<191 and not row[col], 'Financial cell collision.'
                row[col]=value
            assert re.fullmatch(r'\d+',row[0])
            records.append(row)
            provenance.append({'Source_Row':len(records),'Transaction_Page':i+1,'Financial_Page':i+45,'PDF_Y':y,'Order_ID':row[0]})
        # Independent ordinary text extraction must reproduce every nonblank cell in order.
        for pg, side, cols in [(reader.pages[i],left,7),(reader.pages[i+44],right,2)]:
            standard=re.sub(r'\s+','',pg.extract_text())
            rebuilt=''.join(v for y in sorted(side,reverse=True) for x,v in sorted(side[y]))
            if i==0: rebuilt=''.join(columns[:7] if cols==7 else columns[7:])+rebuilt
            assert re.sub(r'\s+','',rebuilt)==standard,'Independent text cross-check failed.'
        audit.append({'Transaction_Page':i+1,'Financial_Page':i+45,'Rows':len(left),'Coordinate_Match':True,'Text_Crosscheck':True})
    raw=pd.DataFrame(records,columns=columns).replace('',pd.NA)
    # Exact duplicates should also repeat the same financial cells, not shifted neighbours.
    left_dups=raw.duplicated(subset=columns[:7]); full_dups=raw.duplicated()
    assert left_dups.equals(full_dups), 'Duplicate fingerprints do not align.'
    return raw, pd.DataFrame(provenance), pd.DataFrame(audit), hashlib.sha256(path.read_bytes()).hexdigest()


In [ ]:
# Locate the exact provided PDF; the last path supports this hosted workspace only.
PDF_NAME = 'AI + Data_ Make Data Intelligent _ Masterclass 1 _ Practice Dataset.pdf'
candidates = [ROOT/'data'/PDF_NAME, ROOT/PDF_NAME, ROOT.parent/'uploads'/PDF_NAME]
PDF_PATH = next((p for p in candidates if p.exists()), None)
if PDF_PATH is None:
    raise FileNotFoundError('Place the official practice PDF in data/ or beside this notebook.')
extracted, provenance, extraction_audit, source_hash = extract_official_pdf(PDF_PATH)
provenance.to_csv(OUT/'pdf_row_provenance.csv', index=False)
extraction_audit.to_csv(OUT/'pdf_extraction_audit.csv', index=False)
# Reconstructed Excel is a local derivative, not the original internship workbook.
workbook = ROOT/'data'/'official_practice_reconstructed.xlsx'
extracted.to_excel(workbook, sheet_name='Reconstructed_Transactions', index=False)
excel = pd.ExcelFile(workbook)
print('DERIVED workbook sheets:', excel.sheet_names)
required = set(extracted.columns)
matching = [s for s in excel.sheet_names if required.issubset(pd.read_excel(excel, sheet_name=s, nrows=0).columns)]
assert len(matching) == 1, 'Select a sheet explicitly after inspection.'
selected_sheet = matching[0]
# Preserve source text and identifiers during the Excel round trip.
raw = pd.read_excel(excel, sheet_name=selected_sheet, dtype='string')
pd.testing.assert_frame_equal(raw.fillna('').astype(str), extracted.fillna('').astype(str), check_dtype=False)
raw_before = raw.copy(deep=True)
print('Selected derived sheet:', selected_sheet, '| PDF SHA-256:', source_hash)
display(extraction_audit)
print('All paired page checks passed; reconstructed rows:', len(raw))

## 04 Dataset Inspection
Dtypes below describe the reconstructed text-preserving import, not original Excel cell types. Valid date coverage excludes unparseable date strings, which are counted separately.

In [ ]:
# Inspect without modifying raw data.
display(raw.head(5)); display(raw.tail(5))
print('Shape:', raw.shape, '\nActual columns:', raw.columns.tolist())
display(raw.dtypes.rename('Imported_Dtype').to_frame())
display(raw.isna().sum().rename('Missing').to_frame())
print('Exact duplicate records:', int(raw.duplicated().sum()))
display(raw.describe(include='all').T)
inspection_dates = pd.to_datetime(raw['Order_Date'], format='%d-%m-%Y', errors='coerce')
print('Valid date range:', inspection_dates.min(), 'to', inspection_dates.max())
print('Invalid dates:', int(inspection_dates.isna().sum()))
print('Unique customers:', raw['Customer_ID'].nunique(), '| Unique orders:', raw['Order_ID'].nunique())
for col in ['Product', 'Category', 'Region']:
    print(col, sorted(raw[col].dropna().unique().tolist()))
print('Numerical business fields observed in PDF headers: Quantity, Revenue, Profit')

## 05 Data Quality
**Policy before analysis:**
- Remove only complete exact duplicates; distinct lines are not dropped on Order_ID alone.
- Normalize Category/Region case and whitespace; preserve identifier spelling and product names.
- Parse explicit day-month-year dates; impossible dates become missing, never guessed.
- Strip only observed currency tokens (`INR`, `₹`, mojibake `â‚¹`) and commas. Unknown formats stop execution.
- Missing product/region gets an explicit `Unknown` category. Missing money is not replaced with zero; no financial values are invented.
- Non-positive quantity and negative revenue are quarantined from purchase/sales analysis because return/error semantics are unavailable. Negative profit can represent a real loss and is retained.
- IQR outliers are flagged, not automatically removed: expensive products naturally have high revenue.
- Valid transactions with missing Customer_ID still contribute to sales EDA but cannot enter customer analysis. Missing revenue transactions still count as purchase events if their dates/IDs are valid.
- A customer with any invalid-date record is excluded from the predictive cohort: otherwise their purchase could be missed in the outcome window. This conservative exclusion can introduce selection bias.

Every action's affected count is logged; audit counts can overlap and are not additive.

In [ ]:
# Profile suspicious values before deciding whether to retain or quarantine.
def parse_number(series):
    text = series.astype('string').str.strip()
    for token in ['â‚¹', '₹', 'INR', ',']:
        text = text.str.replace(token, '', regex=False)
    text = text.str.strip()
    invalid = text.notna() & ~text.str.fullmatch(r'-?\d+(?:\.\d+)?', na=False)
    assert not invalid.any(), 'Review unrecognized numeric values: ' + str(series[invalid].tolist())
    return pd.to_numeric(text, errors='coerce').astype(float)
quality = pd.DataFrame({'Missing_Raw': raw.isna().sum()})
display(quality)
for col in ['Quantity', 'Revenue', 'Profit']:
    numeric = parse_number(raw[col])
    print(col, 'negative:', int(numeric.lt(0).sum()), 'zero:', int(numeric.eq(0).sum()))
print('Invalid date strings:', raw.loc[inspection_dates.isna(), 'Order_Date'].value_counts().to_dict())

## 06 Data Cleaning
A preserved cleaned master table supports separate sales, purchase-event and customer views. Rows omitted from a particular analysis remain in the audit files. Quantity charts explicitly omit unknown quantity, and profit charts show observed profit only.

In [ ]:
# Clean a copy; log transformations, preserve original values and row lineage.
changes = []
def log(action, affected, reason):
    changes.append({'Action':action, 'Affected':int(affected), 'Reason':reason})
clean = raw.loc[~raw.duplicated()].copy()
log('Exact duplicates removed', raw.duplicated().sum(), 'All nine source fields identical; prevent double counting.')
assert not clean['Order_ID'].duplicated().any(), 'Review multiple distinct lines per order before interpreting grain.'
clean['Order_Date'] = pd.to_datetime(clean['Order_Date'], format='%d-%m-%Y', errors='coerce')
log('Invalid dates set to NaT', clean['Order_Date'].isna().sum(), 'Impossible calendar dates cannot be safely inferred.')
for col in ['Category', 'Region']:
    normalized = clean[col].str.strip().str.title()
    log(col+' case/space normalization', (normalized.fillna('') != clean[col].fillna('')).sum(), 'Merge equivalent labels only.')
    clean[col] = normalized
for col in ['Product', 'Region']:
    log(col+' missing labelled Unknown', clean[col].isna().sum(), 'Keep valid revenue; make missing dimension visible.')
    clean[col] = clean[col].fillna('Unknown')
for col in ['Quantity','Revenue','Profit']:
    log(col+' nonmissing text values parsed', clean[col].notna().sum(), 'Convert reconstructed text to numeric for arithmetic.')
    clean[col] = parse_number(clean[col])
# Quarantine dubious purchases, not valid negative-profit orders.
suspect_purchase = clean['Quantity'].le(0) | clean['Revenue'].lt(0)
log('Suspect purchases quarantined', suspect_purchase.sum(), 'Return versus data error cannot be resolved; no sign-flipping.')
log('Negative profit retained', clean['Profit'].lt(0).sum(), 'A loss is possible; not inherently an error.')
clean['Month'] = clean['Order_Date'].dt.to_period('M').astype('string')
clean['Year'] = clean['Order_Date'].dt.year
sales = clean.loc[clean['Order_Date'].notna() & clean['Revenue'].notna() & ~suspect_purchase].copy()
events = clean.loc[clean['Order_Date'].notna() & clean['Customer_ID'].notna() & ~suspect_purchase].copy()
log('Sales-view rows excluded', len(clean)-len(sales), 'Invalid date, absent revenue or suspect purchase; retained in cleaned master.')
log('Customer-event rows excluded', len(clean)-len(events), 'Invalid date, absent customer ID or suspect purchase; missing revenue allowed.')
invalid_date_ids = set(clean.loc[clean['Order_Date'].isna(), 'Customer_ID'].dropna())
# Outlier flags are descriptive and do not influence exclusions or the ML pipeline.
outliers = []
for col in ['Quantity','Revenue','Profit']:
    q1,q3 = clean[col].quantile([.25,.75]); iqr=q3-q1
    flags = (clean[col] < q1-1.5*iqr) | (clean[col] > q3+1.5*iqr)
    outliers.append({'Column':col, 'Lower_IQR_Fence':q1-1.5*iqr, 'Upper_IQR_Fence':q3+1.5*iqr, 'Flagged':int(flags.sum()), 'Action':'Retain; review'})
cleaning_audit = pd.DataFrame(changes)
display(cleaning_audit); display(pd.DataFrame(outliers)); display(clean.describe())
cleaning_audit.to_csv(OUT/'cleaning_audit.csv',index=False)
pd.DataFrame(outliers).to_csv(OUT/'outlier_audit.csv',index=False)
clean.to_csv(OUT/'cleaned_transactions.csv',index=False)
clean.loc[~clean.index.isin(sales.index)].to_csv(OUT/'sales_quarantine.csv',index=False)
raw.isna().sum().rename('Missing').to_csv(OUT/'raw_missing_values.csv')
pd.testing.assert_frame_equal(raw, raw_before)
print('Raw input unchanged. Clean master:', clean.shape, '| Sales view:', sales.shape, '| Customer events:', events.shape)

## 07 Exploratory Data Analysis
Revenue and order KPIs use the valid **sales view**, while customer behaviour uses **purchase events** (including purchases with missing amounts). They have different denominators, reported explicitly. INR is indicated by the source's currency tokens. The final August period is incomplete; a lower August total is not evidence of a genuine business decline.

Charts answer month, category, region, product and customer questions. Lowest revenue means lowest **observed contribution**, not failure relative to a target, market size or profitability. No market-potential data were provided.

In [ ]:
# Aggregate actual sales. Missing profit is not treated as zero in totals.
monthly = sales.groupby('Month', observed=True)['Revenue'].sum().sort_index()
category = sales.groupby('Category', observed=True)['Revenue'].sum().sort_values(ascending=False)
region = sales.groupby('Region', observed=True)['Revenue'].sum().sort_values(ascending=False)
products = sales.groupby('Product', observed=True)['Revenue'].sum().sort_values(ascending=False)
customer_sales = sales.dropna(subset=['Customer_ID']).groupby('Customer_ID')['Revenue'].sum().sort_values(ascending=False)
quantity = sales.dropna(subset=['Quantity']).groupby('Category')['Quantity'].sum().sort_values(ascending=False)
profit_category = sales.groupby('Category')['Profit'].sum(min_count=1).sort_values(ascending=False)
profit_region = sales.groupby('Region')['Profit'].sum(min_count=1).sort_values(ascending=False)
order_counts = events.groupby('Customer_ID')['Order_ID'].nunique()
customer_distribution = pd.Series({'One-time':int(order_counts.eq(1).sum()), 'Repeat':int(order_counts.gt(1).sum())})
# Helper enforces titles and both axis labels on every chart.
def save_bar(series, title, xlabel, ylabel, filename, horizontal=False):
    fig,ax=plt.subplots(figsize=(9,4.8))
    series.plot(kind='barh' if horizontal else 'bar', ax=ax, color='#177e89')
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    if not horizontal: ax.tick_params(axis='x', rotation=25)
    fig.tight_layout(); fig.savefig(CHARTS/filename, dpi=140); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(9,4.8)); ax.plot(monthly.index,monthly.values,marker='o',color='#177e89')
ax.set(title='Monthly observed revenue (August incomplete)',xlabel='Order month',ylabel='Revenue (INR)')
ax.tick_params(axis='x',rotation=25);fig.tight_layout();fig.savefig(CHARTS/'revenue_by_month.png',dpi=140);plt.show();plt.close(fig)
save_bar(category,'Revenue by category','Category','Revenue (INR)','revenue_by_category.png')
save_bar(region,'Revenue by region (city labels)','Region','Revenue (INR)','revenue_by_region.png')
save_bar(products.head(10).sort_values(),'Top 10 products by revenue','Revenue (INR)','Product','top_products.png',True)
save_bar(customer_sales.head(10).sort_values(),'Top 10 customers by observed revenue','Revenue (INR)','Coded customer ID','top_customers.png',True)
save_bar(quantity,'Known quantity by category','Category','Units','quantity_by_category.png')
save_bar(profit_category,'Observed profit by category','Category','Profit (INR)','profit_by_category.png')
save_bar(profit_region,'Observed profit by region','Region','Profit (INR)','profit_by_region.png')
save_bar(customer_distribution,'Customers: one-time versus repeat purchases','Customer segment','Customers','customer_distribution.png')
fig,ax=plt.subplots(figsize=(9,4.8)); sns.histplot(sales['Revenue'],bins=40,ax=ax,color='#177e89')
ax.set(title='Transaction revenue distribution (outliers retained)',xlabel='Revenue per transaction (INR)',ylabel='Transactions')
fig.tight_layout();fig.savefig(CHARTS/'revenue_distribution.png',dpi=140);plt.show();plt.close(fig)
# Save auditable aggregates used by the report and dashboard.
for name,series in [('monthly_revenue',monthly),('category_revenue',category),('region_revenue',region),('product_revenue',products),('customer_revenue',customer_sales)]:
    series.to_csv(OUT/(name+'.csv'))
display(monthly.rename('Revenue').to_frame()); display(category.rename('Revenue').to_frame())
print('Most frequent customers:'); display(order_counts.sort_values(ascending=False).head(10).rename('Orders').to_frame())
print('Missing profit within sales view:', int(sales['Profit'].isna().sum()), '| Missing quantity:',int(sales['Quantity'].isna().sum()))

## 08 Customer-Level Dataset
**One customer = one row.** Frequency counts distinct orders, not line items. AOV is total known revenue divided by distinct orders **only when all revenue values are known**; otherwise it is missing. `Total_Revenue` is an observed subtotal, accompanied by `Revenue_Missing_Orders`. This avoids presenting incomplete revenue as a complete customer total.

In [ ]:
# Reusable customer aggregation; columns correspond to the actual source headers.
def aggregate_customers(frame):
    grouped = frame.groupby('Customer_ID', observed=True)
    result = grouped.agg(Order_Count=('Order_ID','nunique'),
        Total_Revenue=('Revenue', lambda x:x.sum(min_count=1)),
        First_Purchase=('Order_Date','min'), Last_Purchase=('Order_Date','max'),
        Revenue_Missing_Orders=('Revenue',lambda x:x.isna().sum())).reset_index()
    result['Avg_Order_Value'] = result['Total_Revenue']/result['Order_Count']
    result.loc[result['Revenue_Missing_Orders'].gt(0),'Avg_Order_Value'] = np.nan
    assert result['Customer_ID'].is_unique
    return result
customers = aggregate_customers(events)
display(customers.head(10)); print('Total customers:',len(customers));print('Columns:',customers.columns.tolist())

## 09 RFM Analysis
- **Recency:** days since the most recent purchase; higher values indicate a longer gap.
- **Frequency:** distinct purchases in the observed period.
- **Monetary:** total purchase revenue in the period, missing if any purchase amount is unknown.

Full-period RFM uses the latest valid source date as reference. It is descriptive only and is **not** passed to the historical model. Historical RFM will be recalculated at the cutoff.

In [ ]:
# Descriptive RFM, kept separate from predictive historical features.
reference_date = clean['Order_Date'].max()
def add_rfm(frame, reference):
    result=frame.copy()
    result['Recency']=(reference-result['Last_Purchase']).dt.days
    result['Frequency']=result['Order_Count']
    result['Monetary']=result['Total_Revenue'].where(result['Revenue_Missing_Orders'].eq(0))
    return result
customers=add_rfm(customers,reference_date)
customers['Customer_Segment']=np.where(customers['Frequency'].eq(1),'One-time','Repeat')
customers.to_csv(OUT/'customer_rfm_full_period.csv',index=False)
print('Full-period reference date:',reference_date.date());display(customers.head(10))

## 10 Churn Definition
### Resolving the workbook inconsistency
Masterclass 3 p.12 gives both (a) Recency ≥90 days and (b) inactivity ≥180 days **and** Frequency <5 **and** Monetary <100,000. Page 15 separately recommends a historical snapshot with 180-day inactivity. These definitions are not equivalent.

**Selected target:** among eligible customers with at least one valid purchase on or before the cutoff, `Churn_Status=1` if there is **no recorded valid purchase in (cutoff, cutoff + 180 days]**; otherwise 0. It is a *future 180-day inactivity proxy*, not confirmed permanent departure. The frequency/monetary restrictions are not used because they describe low-value customers rather than an observed future purchase outcome.

The final valid date defines the observation end; cutoff = end −180 days. This uses real coverage rather than inventing later records. The short historical window and absence of a source completeness guarantee are limitations. We assume the export covers the interval continuously; no event log proves that assumption. A coverage check is shown below.

Customers first appearing after the cutoff are excluded from prediction; customers with invalid-date records are excluded to reduce outcome ambiguity. No threshold is changed after looking at model performance. Classroom labels are computed below **only for comparison**, never as model input.

In [ ]:
# Define historical and future periods before any model fitting.
HORIZON_DAYS = 180
cutoff = reference_date - pd.Timedelta(days=HORIZON_DAYS)
eligible_events = events.loc[~events['Customer_ID'].isin(invalid_date_ids)].copy()
historical = eligible_events.loc[eligible_events['Order_Date'].le(cutoff)].copy()
future = eligible_events.loc[eligible_events['Order_Date'].gt(cutoff) & eligible_events['Order_Date'].le(reference_date)].copy()
assert not historical.empty and (reference_date-cutoff).days==180
snapshot=add_rfm(aggregate_customers(historical),cutoff)
future_buyers=set(future['Customer_ID'])
snapshot['Churn_Status']=(~snapshot['Customer_ID'].isin(future_buyers)).astype(int)
assert snapshot['Churn_Status'].nunique()==2, 'Cannot train binary logistic regression on a single-class target.'
classroom_counts={
    'Descriptive 90-day inactivity':int(customers['Recency'].ge(90).sum()),
    'Descriptive 180-day AND low frequency AND low monetary (known amounts only)':int((customers['Recency'].ge(180)&customers['Frequency'].lt(5)&customers['Monetary'].lt(100000)).sum())}
print('Historical:',historical['Order_Date'].min().date(),'to',cutoff.date(),'(inclusive)')
print('Outcome:',(cutoff+pd.Timedelta(days=1)).date(),'to',reference_date.date(),'(inclusive)')
print('History length in calendar days:',(cutoff-historical['Order_Date'].min()).days+1)
print('Predictive cohort:',len(snapshot),'| Outcome counts:',snapshot['Churn_Status'].value_counts().to_dict())
print('Customers with invalid-date records excluded:',len(invalid_date_ids))
print('Classroom comparisons, NOT predictive targets:',classroom_counts)
print('Coverage by month (all valid dated rows):')
display(clean.dropna(subset=['Order_Date']).groupby('Month').size().rename('Rows').to_frame())
print('New customers only seen after cutoff:',len(set(eligible_events['Customer_ID'])-set(snapshot['Customer_ID'])))
snapshot.to_csv(OUT/'historical_customer_dataset.csv',index=False)

## 11 TARGET LEAKAGE CHECK
Target leakage gives the model information unavailable at the intended prediction time. Using full-period Recency to predict a label directly defined by that same Recency would be circular.

Here, feature events are dated **on/before cutoff**; the label depends only on purchases **after cutoff**. Historical Recency is therefore not the same quantity as future inactivity. Customer_ID, all full-period aggregates, outcome-window counts and Churn_Status are excluded from the feature matrix. Preprocessing learns medians and scaling from training rows only.

Invalid-date exclusion is a retrospective data-quality screen, not an input feature. It may cause selection bias. Random splitting separates customers at one fixed snapshot; it does **not** establish generalization to a later calendar period. Production deployment requires temporal backtesting and independent data-quality monitoring.

In [ ]:
# Executable leakage and aggregation checks.
FEATURES=['Recency','Frequency','Monetary','Avg_Order_Value']
assert historical['Order_Date'].max()<=cutoff
assert future['Order_Date'].min()>cutoff
assert not set(historical.index)&set(future.index)
assert snapshot['Last_Purchase'].max()<=cutoff
assert snapshot['Customer_ID'].is_unique
assert 'Customer_ID' not in FEATURES and 'Churn_Status' not in FEATURES
assert snapshot['Frequency'].eq(snapshot['Order_Count']).all()
assert snapshot['Recency'].ge(0).all()
print('PASS: event-time boundary, customer uniqueness, feature allowlist and RFM checks.')

## 12 Feature Selection
Use historical Recency, Frequency, Monetary and Avg_Order_Value only. Monetary and AOV can be missing because incomplete monetary history is not invented. Correlated RFM features limit coefficient interpretation; association is not causation.

In [ ]:
# Explicit allowlist: no identifier or target-derived field can enter X.
X=snapshot[FEATURES].copy()
y=snapshot['Churn_Status'].copy()
display(X.isna().sum().rename('Missing_Features').to_frame())
assert not X.isna().all().any(), 'An entirely missing feature needs separate treatment.'
assert np.isfinite(X.fillna(0).to_numpy()).all()
display(X.describe())

## 13 Train/Test Split
A reproducible 75%/25% customer split uses stratification to preserve the minority class. Test customers are never used to fit medians, scaling or logistic coefficients. No hyperparameter tuning is performed against the test set.

In [ ]:
# Split customer row indices so identifiers can be reattached only after prediction.
train_idx,test_idx=train_test_split(snapshot.index,test_size=.25,random_state=SEED,stratify=y)
assert not set(train_idx)&set(test_idx)
X_train,X_test=X.loc[train_idx],X.loc[test_idx]
y_train,y_test=y.loc[train_idx],y.loc[test_idx]
print('Training customers:',len(train_idx),'| Test customers:',len(test_idx))
print('Training labels:',y_train.value_counts().to_dict(),'| Test labels:',y_test.value_counts().to_dict())

## 14 Logistic Regression
Logistic Regression estimates a probability between zero and one. `SimpleImputer` fills missing features using training medians, `StandardScaler` puts features on comparable scales, and L2-regularized Logistic Regression learns associations. Default class weights are retained; oversampling is not used. A 0.50 probability threshold is used for binary predictions, distinct from the classroom's risk bands.

A majority-class baseline is included so class imbalance cannot make a weak model look useful through accuracy alone.

In [ ]:
# Fit preprocessing and classifier as one leakage-safe training pipeline.
model=Pipeline([('imputer',SimpleImputer(strategy='median')),
                ('scaler',StandardScaler()),
                ('logistic',LogisticRegression(max_iter=2000,random_state=SEED))])
model.fit(X_train,y_train)
class_one_index=list(model.classes_).index(1)
test_probability=model.predict_proba(X_test)[:,class_one_index]
test_prediction=(test_probability>=.5).astype(int)
baseline=DummyClassifier(strategy='most_frequent').fit(X_train.fillna(0),y_train)
baseline_prediction=baseline.predict(X_test.fillna(0))
print('Model fitted. Convergence iterations:',model.named_steps['logistic'].n_iter_)
print('Training medians:',dict(zip(FEATURES,model.named_steps['imputer'].statistics_)))

## 15 Model Evaluation
All metrics here use the held-out customers only. Positive means future inactivity. Precision measures correctness among flagged customers; recall measures the proportion of actual inactive customers found. ROC-AUC measures ranking, not probability calibration. Zero-division metrics are reported as 0 and must be interpreted alongside predicted-positive counts.

In [ ]:
# Calculate actual held-out metrics and majority baseline.
def metric_row(name,actual,pred,prob=None):
    return {'Model':name,'Accuracy':accuracy_score(actual,pred),
            'Precision':precision_score(actual,pred,zero_division=0),
            'Recall':recall_score(actual,pred,zero_division=0),
            'F1':f1_score(actual,pred,zero_division=0),
            'ROC_AUC':roc_auc_score(actual,prob) if prob is not None and actual.nunique()==2 else np.nan}
metrics=pd.DataFrame([metric_row('Logistic Regression',y_test,test_prediction,test_probability),
                      metric_row('Majority baseline',y_test,baseline_prediction)])
metrics.to_csv(OUT/'model_metrics.csv',index=False)
display(metrics)
report_text=classification_report(y_test,test_prediction,labels=[0,1],target_names=['Returned','Inactive'],zero_division=0)
print(report_text); (OUT/'classification_report.txt').write_text(report_text)
cm=confusion_matrix(y_test,test_prediction,labels=[0,1])
pd.DataFrame(cm,index=['Actual returned','Actual inactive'],columns=['Predicted returned','Predicted inactive']).to_csv(OUT/'confusion_matrix.csv')
fig,ax=plt.subplots(figsize=(6.2,4.8));ConfusionMatrixDisplay(cm,display_labels=['Returned','Inactive']).plot(ax=ax,colorbar=False,cmap='Blues')
ax.set(title='Held-out confusion matrix (threshold 0.50)',xlabel='Predicted status',ylabel='Actual future status')
fig.tight_layout();fig.savefig(CHARTS/'confusion_matrix.png',dpi=140);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(6.2,4.8));RocCurveDisplay.from_predictions(y_test,test_probability,ax=ax)
ax.plot([0,1],[0,1],'--',color='grey');ax.set(title='Held-out ROC curve',xlabel='False positive rate',ylabel='True positive rate')
fig.tight_layout();fig.savefig(CHARTS/'roc_curve.png',dpi=140);plt.show();plt.close(fig)
print('Predicted inactive:',int(test_prediction.sum()),'| Actual inactive:',int(y_test.sum()))
print('Brier score:',brier_score_loss(y_test,test_probability),'(not a full calibration study)')
print('TN, FP, FN, TP:',cm.ravel().tolist())

## 16 Customer Predictions
The held-out prediction table is the unbiased evaluation output. A second table scores every customer in the **historical snapshot cohort**, explicitly marking Training versus Held-out. Training-row scores are in-sample and must not be treated as evidence of predictive performance. These are retrospective February-cutoff scores, not live August predictions; no future actual labels are available for a new live forecast.

In [ ]:
# Attach customer identifiers AFTER model inference, never as model features.
heldout=snapshot.loc[test_idx,['Customer_ID','Churn_Status']].rename(columns={'Churn_Status':'Actual_Churn_Status'}).copy()
heldout['Predicted_Churn_Status']=test_prediction
heldout['Churn_Probability']=test_probability
heldout['Prediction_Set']='Held-out'
heldout.to_csv(OUT/'heldout_predictions.csv',index=False)
all_predictions=snapshot[['Customer_ID','Churn_Status']].rename(columns={'Churn_Status':'Actual_Churn_Status'}).copy()
all_predictions['Churn_Probability']=model.predict_proba(X)[:,class_one_index]
all_predictions['Predicted_Churn_Status']=(all_predictions['Churn_Probability']>=.5).astype(int)
all_predictions['Prediction_Set']=np.where(snapshot.index.isin(test_idx),'Held-out','Training (in-sample)')
all_predictions.to_csv(OUT/'customer_predictions.csv',index=False)
display(heldout.head(10))

## 17 Risk Levels
Classroom thresholds: probability <40% = Low Risk; 40% to <70% = Medium Risk; ≥70% = High Risk. Empty bands are valid findings, not a reason to change thresholds. The 0.50 prediction threshold and risk categories serve different purposes.

In [ ]:
# Apply exact boundary rules, then sort descending by actual model probability.
risk=snapshot[['Customer_ID']+FEATURES].merge(all_predictions,on='Customer_ID',validate='one_to_one')
risk['Risk_Level']=np.select([risk['Churn_Probability']<.4,risk['Churn_Probability']<.7],['Low Risk','Medium Risk'],default='High Risk')
risk=risk.sort_values(['Churn_Probability','Customer_ID'],ascending=[False,True]).reset_index(drop=True)
risk.to_csv(OUT/'customer_risk_table.csv',index=False)
risk_counts=risk['Risk_Level'].value_counts().reindex(['Low Risk','Medium Risk','High Risk'],fill_value=0)
display(risk_counts.rename('Customers').to_frame())
assert risk['Churn_Probability'].between(0,1).all()
assert np.select([np.array([.399,.4,.699,.7])<.4,np.array([.399,.4,.699,.7])<.7],['Low Risk','Medium Risk'],default='High Risk').tolist()==['Low Risk','Medium Risk','Medium Risk','High Risk']

## 18 Top 20 Highest-Risk Customers
“Top 20” means the 20 highest scores, not necessarily 20 customers in the ≥70% High Risk band. Customer IDs are coded identifiers only. The prediction-set column makes in-sample status visible.

In [ ]:
# Retain actual probabilities; percentage formatting is presentation only.
top20=risk.head(20).copy()
top20.to_csv(OUT/'top20_risk_customers.csv',index=False)
display(top20.style.format({'Churn_Probability':'{:.2%}','Monetary':'{:,.0f}','Avg_Order_Value':'{:,.0f}'}))

## 19 Risk Analysis
Compare top-20 medians with the full snapshot cohort. A direction is reported only if the values support it. Associations with model scores are not proof of a causal mechanism or independently validated churn drivers.

In [ ]:
# Generate evidence-grounded observations, not assumed customer motivations.
risk_comparison=pd.DataFrame({'Top20_Median':top20[FEATURES].median(),'Cohort_Median':snapshot[FEATURES].median()})
risk_comparison.to_csv(OUT/'risk_comparison.csv')
display(risk_comparison)
risk_observations=[f"Top 20 probabilities range from {top20.Churn_Probability.min():.2%} to {top20.Churn_Probability.max():.2%}.",
 f"The top 20 contain {int(top20.Risk_Level.eq('High Risk').sum())} High Risk customers and {int(top20.Prediction_Set.eq('Held-out').sum())} held-out customers.",
 f"{int(top20.Frequency.eq(1).sum())} of the top 20 made only one historical purchase."]
for feature in FEATURES:
    a,b=risk_comparison.loc[feature]
    direction='higher than' if a>b else 'lower than' if a<b else 'equal to'
    risk_observations.append(f'{feature}: top-20 median {a:,.2f} is {direction} cohort median {b:,.2f}.')
possible_insights=[]
for feature in ['Recency','Frequency','Monetary']:
    a,b=risk_comparison.loc[feature]
    if a!=b:
        possible_insights.append(f"{'Higher' if a>b else 'Lower'} {feature} is visible among higher-scored customers in this sample; this may guide further investigation, not establish causation.")
print('OBSERVATIONS');print('\n'.join(risk_observations))
print('POSSIBLE INSIGHTS');print('\n'.join(possible_insights))

## 20 Dashboard
Self-contained Plotly HTML embeds JavaScript (no CDN) so it works offline. Hover, zoom and legend interactions are supported. Cohort labels separate full-period sales/customer KPIs from the historical predictive cohort. The high-risk table uses **only ≥70%** customers; an empty band is shown explicitly. RFM charts describe the historical cohort; missing monetary amounts are omitted, not fabricated.

In [ ]:
# Build offline interactive figures and explicitly labelled KPI cards.
def prep(fig,title,x,y):
    fig.update_layout(title=title,xaxis_title=x,yaxis_title=y,template='plotly_white',margin=dict(l=55,r=25,t=65,b=55),height=350,font=dict(family='Arial'))
    return fig
figures=[]
figures.append(prep(px.line(x=monthly.index,y=monthly.values,markers=True),'Revenue trend · August incomplete','Month','Revenue (INR)'))
figures.append(prep(px.bar(x=category.index,y=category.values),'Category performance','Category','Revenue (INR)'))
figures.append(prep(px.bar(x=region.index,y=region.values),'Region performance','Region','Revenue (INR)'))
figures.append(prep(px.bar(x=risk_counts.index,y=risk_counts.values,color=risk_counts.index,color_discrete_map={'Low Risk':'#1b998b','Medium Risk':'#e9a23b','High Risk':'#d1495b'}),'Historical customer risk bands','Risk band','Customers'))
figures.append(prep(px.histogram(risk,x='Churn_Probability',nbins=20),'Historical churn probabilities','Estimated probability','Customers'))
for f in ['Recency','Frequency','Monetary']:
    figures.append(prep(px.histogram(snapshot,x=f,nbins=25),f+' · historical snapshot',f+(' (INR)' if f=='Monetary' else ' (days)' if f=='Recency' else ' (orders)'),'Customers'))
high=risk.loc[risk.Risk_Level.eq('High Risk'),['Customer_ID','Recency','Frequency','Monetary','Churn_Probability','Prediction_Set']].head(20).copy()
if len(high):
    high['Churn_Probability']=high['Churn_Probability'].map(lambda p:f'{p:.2%}')
    table_html=high.to_html(index=False,classes='risk-table',float_format=lambda n:f'{n:,.0f}')
else: table_html='<p>No customer meets the unchanged 70% High Risk threshold.</p>'
kpis=[('Total Customers',f'{len(customers):,}','Full-period valid customer events'),
      ('Total Revenue',f'INR {sales.Revenue.sum():,.0f}','Sales view · observed amounts'),
      ('Total Orders',f'{sales.Order_ID.nunique():,}','Sales view · distinct orders'),
      ('Average Order Value',f'INR {sales.Revenue.sum()/sales.Order_ID.nunique():,.2f}','Sales view · known amounts'),
      ('Churned Customers',str(int(y.sum())),f'Historical cohort n={len(snapshot)} · 180-day inactivity'),
      ('Churn Rate',f'{y.mean():.2%}','Historical cohort · future inactivity proxy'),
      ('High Risk Customers',str(int(risk_counts["High Risk"])),f'Historical cohort · model probability ≥70%')]
cards=''.join(f'<div class="card"><small>{label}</small><strong>{value}</strong><span>{note}</span></div>' for label,value,note in kpis)
plots=''.join('<section>'+f.to_html(full_html=False,include_plotlyjs=False,config={'responsive':True,'displaylogo':False})+'</section>' for f in figures)
html="""<!DOCTYPE html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Rohit Dey | Customer Risk Analytics</title><style>
body{margin:0;background:#f0f4f8;color:#142b40;font-family:Arial,sans-serif}header{background:#102d43;color:white;padding:32px 5%}header p{color:#bce1e6;max-width:1000px;line-height:1.6}main{max-width:1400px;margin:auto;padding:24px}.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px}.card,section,.note{background:white;border-radius:10px;padding:18px;box-shadow:0 3px 12px #162e4010}.card small{font-size:13px}.card strong{display:block;font-size:25px;margin:12px 0;color:#127986}.card span{font-size:12px;color:#5e6c78}.grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:16px;margin:22px 0}.note{line-height:1.65;margin:18px 0}table{border-collapse:collapse;width:100%;font-size:13px}td,th{padding:10px;text-align:left;border-bottom:1px solid #e1e9ee}footer{padding:25px;text-align:center;color:#657583}@media(max-width:800px){.grid{grid-template-columns:1fr}.risk-table{font-size:10px}}
</style></head><body><header><small>AICTE | IBM SKILLSBUILD · BHARATCARES · ACADEMIC PROJECT</small><h1>Customer Churn & Risk Analytics</h1><p>Rohit Dey · RFM + Logistic Regression · Official practice dataset PDF reconstruction</p></header><main>"""
html+='<div class="kpis">'+cards+'</div>'
html+=f'<div class="note"><b>Two scopes:</b> sales/customer KPIs cover {events.Order_Date.min().date()}–{reference_date.date()}; model scores describe {len(snapshot)} customers known at {cutoff.date()}. Labels observe no purchase during the following 180 days. Training-row scores are in-sample. This is a retrospective teaching experiment, not a validated live retention system.</div>'
html+='<script>'+get_plotlyjs()+'</script><div class="grid">'+plots+'</div>'
html+='<section><h2>High Risk customers · ≥70% (up to 20)</h2>'+table_html+'</section>'
html+='<div class="note"><b>Held-out evaluation:</b> '+metrics.iloc[0].drop('Model').round(4).to_string().replace('\n',' · ')+'<br>Do not use accuracy alone. Probabilities are not guaranteed outcomes or calibrated intervention benefits. Validate future cohorts before automated actions.</div>'
html+='</main><footer>Source: official internship practice PDF · No external scripts or datasets required</footer></body></html>'
(DASH/'customer_churn_dashboard.html').write_text(html,encoding='utf-8')
print('Saved:',DASH/'customer_churn_dashboard.html')
display(pd.DataFrame(kpis,columns=['KPI','Value','Scope']))

## 21 Business Insights
The following statements are generated from executed aggregates. Comparisons exclude the incomplete final month when discussing a complete-month trend. Three hypotheses are explicitly distinguished from verified observations.

In [ ]:
# Evidence-driven insights; no copied workbook numbers.
full_months=monthly.iloc[:-1] if reference_date.day < reference_date.days_in_month else monthly
change=(full_months.iloc[-1]/full_months.iloc[0]-1) if len(full_months)>1 else np.nan
known_regions=region.drop('Unknown', errors='ignore')
observations=[f"Highest revenue month: {monthly.idxmax()} (INR {monthly.max():,.0f}).",
 f"Top product: {products.idxmax()} (INR {products.max():,.0f}).",
 f"Top category: {category.idxmax()} (INR {category.max():,.0f}); lowest contribution: {category.idxmin()} (INR {category.min():,.0f}).",
 f"Top region: {region.idxmax()} (INR {region.max():,.0f}); lowest known-region contribution: {known_regions.idxmin()} (INR {known_regions.min():,.0f}). Unknown is a missing-label bucket, not a real region.",
 f"Highest observed-revenue customer: {customer_sales.idxmax()} (INR {customer_sales.max():,.0f}).",
 f"Highest-frequency customer: {order_counts.idxmax()} ({int(order_counts.max())} orders; ties possible).",
 f"Complete-month endpoint change ({full_months.index[0]} to {full_months.index[-1]}): {change:+.2%}; not necessarily a monotonic trend."]
insights=[f"{category.idxmax()} contributes {category.max()/sales.Revenue.sum():.2%} of observed sales revenue, so category-level planning deserves attention.",
 f"{region.idxmax()} contributes {region.max()/sales.Revenue.sum():.2%} of observed sales revenue; this is not evidence of higher market penetration.",
 f"Repeat customers account for {order_counts.gt(1).mean():.2%} of identifiable purchase-event customers.",
 f"Future 180-day inactivity affects {y.mean():.2%} of the historical cohort, not all full-period customers.",
 f"Logistic held-out accuracy {metrics.iloc[0].Accuracy:.2%} should be compared with majority accuracy {metrics.iloc[1].Accuracy:.2%}; recall is {metrics.iloc[0].Recall:.2%}."]
hypotheses=[
 'Category revenue concentration may reflect product price or volume mix; compare units and margins before changing spend.',
 'Regional differences may reflect customer-base size or order mix; regional population and acquisition data are needed to test this.',
 'Long gaps between purchases may reflect normal replenishment cycles rather than permanent departure; test against longer history and product-specific cadence.']
for title,items in [('OBSERVATIONS',observations),('INSIGHTS',insights),('HYPOTHESES (NOT VERIFIED)',hypotheses)]:
    print(title);print('\n'.join('- '+s for s in items))

## 22 Business Actions and Business Decision
At-risk identification can help allocate a limited communication budget, but prediction is not a guarantee and does not establish that a discount will change behaviour.

**Precision versus recall:** a false positive spends contact/offer budget on a customer who would return anyway; a false negative misses a truly inactive customer. Higher recall may suit expensive missed churn, while higher precision may suit limited campaign capacity. No monetary cost ratio or campaign budget was supplied; do not invent ROI. A classroom “500 contacts” example is not the dataset's actual capacity.

Use probabilities for a *review queue*, not automatic discounts. With weak holdout results, prioritize validation over deployment. Suggested actions below remain experiments, not proven solutions.

In [ ]:
# Recommendations depend on actual risk counts and observed evaluation quality.
actions=[
 (f"Review the {int(risk_counts['High Risk'])} customers in the ≥70% band only after score validation." if risk_counts['High Risk'] else "No customer meets the ≥70% High Risk threshold. Do not relabel lower-scored customers as High Risk to fill a campaign queue."),
 'Pilot consent-based re-engagement on a small reviewed, higher-scored group with a randomized no-contact control; measure incremental repeat purchases and opt-outs.',
 'Within the review queue, consider observed monetary value and margin alongside risk to limit offer cost; do not assume low spend explains departure.',
 'Use known product/category history to test relevant reminders, without assuming customer dissatisfaction or motivation.',
 'Audit invalid dates and missing financial values with the trainer/source owner; then obtain longer history, backtest across cutoffs and assess calibration before live use.'
]
if metrics.iloc[0]['Recall'] < .5:
    actions.insert(0,'The held-out model misses more than half of inactive customers at threshold 0.50; do not automate retention decisions. Improve validation/data first.')
print('\n'.join(f'{i+1}. {text}' for i,text in enumerate(actions)))

## 23 Conclusion and Reproducibility Checks
This project completes the official cleaning → EDA → RFM → prediction → risk → dashboard → action workflow, using only supplied practice data. A leakage-safe time boundary is necessary but not sufficient for business usefulness.

**Limitations:** reconstructed PDF cannot verify original Excel sheets/types/formulas; data are a classroom practice export with no verified public URL or completeness guarantee; short pre-cutoff history and few positive cases; missing/invalid data and conservative exclusions; single-snapshot random customer split; no temporal validation or causal experiment; incomplete August; probabilities not fully calibrated; in-sample rows exist in the all-customer risk table. Historical inactivity is not permanent churn. AOV and Monetary share information and coefficient interpretation is limited.

**Future scope:** verified original workbook, longer history, multiple time cutoffs, calibration and uncertainty evaluation, cost-based thresholds on validation data, product-specific inactivity horizons, and randomized retention experiments. Keep official risk bands unchanged for this submission.

**References:** supplied Masterclass 1 theory PDF (data quality); Masterclass 2 theory PDF (EDA, verified insights); Masterclass 3 workbook pp.12–18, 22–29 (churn conflict, historical features, Logistic Regression, risk bands, decisions); official practice PDF (data and classroom prompts). Supermarket report is a structural reference only. Zoom chat is context, not a lecture transcript or verified dataset source. Submission form screenshots define four file types and upload limits.

In [ ]:
# Export machine-readable facts used to build the report and README; prevent drift.
summary={
 'author':'Rohit Dey','source_filename':PDF_NAME,'source_sha256':source_hash,
 'raw_rows':len(raw),'raw_columns':raw.columns.tolist(),'raw_unique_customers':int(raw.Customer_ID.nunique()),
 'raw_unique_orders':int(raw.Order_ID.nunique()),'duplicates':int(raw.duplicated().sum()),
 'missing_raw':{k:int(v) for k,v in raw.isna().sum().items()},
 'clean_master_rows':len(clean),'sales_rows':len(sales),'event_rows':len(events),
 'total_customers':len(customers),'total_revenue':float(sales.Revenue.sum()),'total_orders':int(sales.Order_ID.nunique()),
 'average_order_value':float(sales.Revenue.sum()/sales.Order_ID.nunique()),
 'date_start':str(clean.Order_Date.min().date()),'date_end':str(reference_date.date()),'cutoff':str(cutoff.date()),
 'history_days':int((cutoff-historical.Order_Date.min()).days+1),'horizon_days':HORIZON_DAYS,
 'cohort_customers':len(snapshot),'churned':int(y.sum()),'churn_rate':float(y.mean()),
 'train_customers':len(train_idx),'test_customers':len(test_idx),'test_positive':int(y_test.sum()),
 'risk_counts':{k:int(v) for k,v in risk_counts.items()},'confusion_matrix':cm.tolist(),
 'metrics':metrics.iloc[0].drop('Model').to_dict(),'baseline_accuracy':float(metrics.iloc[1].Accuracy),
 'classroom_comparison':classroom_counts,'excluded_invalid_date_customers':len(invalid_date_ids),
 'observations':observations,'insights':insights,'hypotheses':hypotheses,
 'risk_observations':risk_observations,'possible_insights':possible_insights,'actions':actions,
 'missing_profit_sales':int(sales.Profit.isna().sum()),'missing_quantity_sales':int(sales.Quantity.isna().sum())}
(OUT/'project_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
# Actual pass/fail checks rather than a pre-filled checklist.
checks={
 'Raw unchanged':raw.equals(raw_before),
 'Excel round trip':len(raw)==len(extracted) and raw.columns.tolist()==extracted.columns.tolist(),
 'Page alignment':bool(extraction_audit.Coordinate_Match.all()),
 'Text crosscheck':bool(extraction_audit.Text_Crosscheck.all()),
 'Customer uniqueness':customers.Customer_ID.is_unique,
 'No duplicate clean orders':clean.Order_ID.is_unique,
 'Time separation':bool(historical.Order_Date.max()<=cutoff and future.Order_Date.min()>cutoff),
 'No split overlap':not bool(set(train_idx)&set(test_idx)),
 'Probabilities bounded':bool(risk.Churn_Probability.between(0,1).all()),
 'Risk sorted':risk.Churn_Probability.is_monotonic_decreasing,
 'Metrics finite':bool(np.isfinite(list(summary['metrics'].values())).all()),
 'Dashboard exists':(DASH/'customer_churn_dashboard.html').exists(),
 'Required charts exist':all((CHARTS/f).exists() for f in ['revenue_by_month.png','revenue_by_category.png','revenue_by_region.png','top_products.png','top_customers.png','quantity_by_category.png','profit_by_category.png','profit_by_region.png','customer_distribution.png','revenue_distribution.png','confusion_matrix.png'])}
assert all(checks.values()), checks
pd.DataFrame({'Check':list(checks),'Passed':list(checks.values())}).to_csv(OUT/'quality_checks.csv',index=False)
display(pd.DataFrame({'Check':list(checks),'Passed':list(checks.values())}))
print('Completed. Report and README must use outputs/project_summary.json, not example metrics.')